# SBND RWM Timing Calibration: Overview (Run 1 + Run 2)

- σ, skewness σR − σL, period and period-fit intercept per time chunk vs time, both runs, with the registry beam conditions

## 1. Setup

- Inputs: `CALIB_DIR` = calibration notebook output folder (`$SBND_RWM_OUT`)

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcdefaults()
from matplotlib.ticker import FormatStrFormatter, MultipleLocator
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

import sys
sys.path.insert(0, '../lib')
from stacking import load_calibration
import event_timestamps_run1 as ev1
import event_timestamps_run2 as ev2
from plot_helpers import CONDITION_HANDLES, CONDITION_HANDLES_RUN2, plot_chunk_series, time_axis

CALIB_DIR = os.environ.get('SBND_RWM_OUT', '../output')   # calibration notebook output: CALIB_DIR/run{1,2}/sbnd_fine_calibration.csv
FIG_DIR = f'{CALIB_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

FULL = 6.3   # figure width [in]
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 7, 'legend.fontsize': 6, 'xtick.labelsize': 7,
                     'ytick.labelsize': 7, 'legend.title_fontsize': 6})

RUNS = {'Run 1': dict(run=1, ev=ev1), 'Run 2': dict(run=2, ev=ev2)}
VALUES = {'sigma': r'$\sigma$ [ns]', 'rl_asym': r'$\sigma_R - \sigma_L$ [ns]', 'period': 'Period [ns]',
          'intercept': 'Period-Fit Intercept [ns]'}
YLIMS = {'sigma': (1.2, 2.8), 'rl_asym': (-0.4, 1.0), 'period': (18.925, 18.945), 'intercept': (510, 545)}
FMT = {'sigma': '%.2f', 'rl_asym': '%.2f', 'period': '%.3f', 'intercept': None}
DROP_LABELS = {'Bunch Rotation Effective', 'Transition Chunk (not segmented)'}
MERGE_LABELS = {'Bunch Rotation Commissioning': 'Bunch Rotation ON / Narrow Bunch', 'LLRF Works': 'Accelerator Works',
                'Water-Pump Offset Steps': 'Accelerator Works', 'Beam Faults / Reduced Beam': 'Accelerator Works'}

## 2. Load Data

In [ ]:
for r in RUNS.values():
    r['df'] = load_calibration(f"{CALIB_DIR}/run{r['run']}/sbnd_fine_calibration.csv")
    print(f"Run {r['run']}: {len(r['df'])} fitted chunks")
RUN2A = (ev2.t00_run2a_start, ev2.t00_run2a_end)   # Production Run 2A (not processed)
SHUTDOWN = (RUNS['Run 1']['df']['end'].max(), RUN2A[0])

## 3. Overview Figure

- Saved to `CALIB_DIR/figures/calib_results_overview.png`

In [ ]:
def overlays(ax):
    ev1.plot_events(ax)
    ev1.plot_extended_downtime(ax)
    for f in (ev2.plot_narrow_bunch, ev2.plot_studies, ev2.plot_beam_faults):
        f(ax)
    ev2.plot_extended_downtime(ax)
    for line in ax.get_lines():
        line.set_linewidth(min(line.get_linewidth(), 0.8))
        if line.get_color() == 'tab:red' and line.get_linestyle() == ':':
            line.set_linestyle('--')
    ax.axvspan(*RUN2A, color='tab:red', alpha=0.1, lw=0)
    ax.axvspan(*SHUTDOWN, color='black', alpha=0.35, lw=0)
    ax.text(SHUTDOWN[0] + (SHUTDOWN[1] - SHUTDOWN[0]) / 2, 0.5, 'Summer 2025\nShutdown', transform=ax.get_xaxis_transform(),
            ha='center', va='center', color='white', fontsize=8, fontweight='bold')


def conditions_handles():
    groups = {}
    for h in CONDITION_HANDLES + CONDITION_HANDLES_RUN2:
        label = MERGE_LABELS.get(h.get_label(), h.get_label())
        if label in DROP_LABELS:
            continue
        g = groups.setdefault(label, [])
        if isinstance(h, Patch):
            if not g:
                g.append(Patch(color=h.get_facecolor()))
        elif label == 'Accelerator Works':
            if not g:
                g.append(Line2D([], [], color='tab:red', linestyle='--', lw=0.8))
        elif (h.get_color(), h.get_linestyle()) not in [(x.get_color(), x.get_linestyle()) for x in g]:
            g.append(Line2D([], [], color=h.get_color(), linestyle=h.get_linestyle(), lw=0.8))
    handles = [tuple(g) if len(g) > 1 else g[0] for g in groups.values()]
    return handles + [Patch(color='tab:red', alpha=0.1)], list(groups) + ['Production Run 2A (not processed)']


fig, axes = plt.subplots(4, 1, figsize=(FULL, 8.4), sharex=True, gridspec_kw=dict(hspace=0.12))
allc = pd.concat([r['df'] for r in RUNS.values()])
for ax, key in zip(axes, VALUES):
    for r in RUNS.values():
        plot_chunk_series(ax, r['df'], key, ev=r['ev'])
    for line in ax.get_lines():
        line.set_linewidth(min(line.get_linewidth(), 0.8))
    if key == 'rl_asym':
        ax.axhline(0, color='k', ls=':', lw=0.8)
    overlays(ax)
    time_axis(ax, (allc['start'].min(), allc['end'].max()))
    ax.set_ylim(*YLIMS[key])
    if FMT[key]:
        ax.yaxis.set_major_formatter(FormatStrFormatter(FMT[key]))
    if key == 'period':
        ax.yaxis.set_major_locator(MultipleLocator(0.005))
    ax.set_ylabel(VALUES[key])
spans = {'Run 1': (RUNS['Run 1']['df']['start'].min(), RUNS['Run 1']['df']['end'].max()),
         'Run 2': (RUN2A[0], RUNS['Run 2']['df']['end'].max())}
for name, (a, b) in spans.items():
    axes[0].text(a + (b - a) / 2, 1.04, f'SBND {name}', transform=axes[0].get_xaxis_transform(), ha='center', va='bottom',
                 fontsize=7, fontweight='bold', bbox=dict(boxstyle='square,pad=0.25', facecolor='white', edgecolor='black', lw=0.6))
handles, labels = conditions_handles()
axes[-1].legend(handles, labels, handler_map={tuple: HandlerTuple(ndivide=None)}, loc='upper center',
                bbox_to_anchor=(0.5, -0.32), ncol=3, frameon=False)
fig.align_ylabels(axes)
fig.savefig(f'{FIG_DIR}/calib_results_overview.png', dpi=300, bbox_inches='tight')
plt.show()